In [1]:
# --- CELL 1: SETUP ---

# 1. FIX PROTOBUF FIRST (Critical fix for AttributeError)
!pip install -q protobuf==3.20.*

# 2. Install your required libraries
!pip install -q transformers==4.44.2 peft==0.12.0 datasets accelerate scikit-learn

# 3. Import libraries
import torch
import numpy as np
import warnings
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import f1_score, accuracy_score

# 4. Filter warnings
warnings.filterwarnings("ignore")

# 5. Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🟢 Device đang dùng: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 3.9 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptograp

2025-12-06 07:16:39.061717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765005399.496036      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765005399.601636      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🟢 Device đang dùng: cuda


In [3]:
# --- CELL 2: DATA MAPPING (Đã sửa lỗi load_dataset) ---
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Định nghĩa Mapping (Theo Paper GoEmotions)
GOEMOTIONS_TO_EKMAN = {
    # Joy Group
    "admiration": "joy", "amusement": "joy", "approval": "joy", "caring": "joy", 
    "desire": "joy", "excitement": "joy", "gratitude": "joy", "joy": "joy", 
    "love": "joy", "optimism": "joy", "pride": "joy", "relief": "joy",
    # Anger Group
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    # Sadness Group
    "sadness": "sadness", "disappointment": "sadness", "embarrassment": "sadness", 
    "grief": "sadness", "remorse": "sadness",
    # Fear Group
    "fear": "fear", "nervousness": "fear",
    # Surprise Group
    "surprise": "surprise", "curiosity": "surprise", "realization": "surprise", "confusion": "surprise",
    # Disgust Group
    "disgust": "disgust",
    # Neutral
    "neutral": "neutral"
}

# 2. Định nghĩa danh sách nhãn mới (7 nhãn)
EKMAN_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
label2id = {l: i for i, l in enumerate(EKMAN_LABELS)}
id2label = {i: l for i, l in enumerate(EKMAN_LABELS)}
NUM_LABELS = len(EKMAN_LABELS)

print(f"🎯 Target Labels: {EKMAN_LABELS}")

# 3. Load & Process Data
print("⏳ Loading GoEmotions...")

# --- SỬA LỖI TẠI ĐÂY ---
try:
    # Thử load bằng tên đầy đủ
    dataset = load_dataset("go_emotions", "simplified")
except Exception:
    # Fallback nếu vẫn lỗi (đôi khi server HF chập chờn)
    print("⚠️ Đang thử load từ mirror hoặc cấu hình mặc định...")
    dataset = load_dataset("goemotions", "simplified")

original_labels = dataset['train'].features['labels'].feature.names

model_checkpoint = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_and_map(examples):
    # Tokenize
    encodings = tokenizer(examples["text"], truncation=True, padding=False, max_length=128)
    
    batch_new_labels = []
    
    for old_ids in examples["labels"]:
        # Tạo vector 0 cho 7 nhãn mới
        new_vec = [0.0] * NUM_LABELS
        
        for old_idx in old_ids:
            old_name = original_labels[old_idx]       # Lấy tên cũ (vd: 'annoyance')
            new_group = GOEMOTIONS_TO_EKMAN[old_name] # Map sang mới (vd: 'anger')
            new_idx = label2id[new_group]             # Lấy ID mới
            new_vec[new_idx] = 1.0                    # Đánh dấu
            
        batch_new_labels.append(new_vec)
        
    encodings["labels"] = batch_new_labels
    return encodings

print("⚙️ Mapping dữ liệu sang nhóm Ekman...")
tokenized_datasets = dataset.map(preprocess_and_map, batched=True, remove_columns=dataset['train'].column_names)
print("✅ Done! Dữ liệu đã sẵn sàng train cho 7 nhóm cảm xúc.")

🎯 Target Labels: ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
⏳ Loading GoEmotions...


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

...

⚙️ Mapping dữ liệu sang nhóm Ekman...


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

✅ Done! Dữ liệu đã sẵn sàng train cho 7 nhóm cảm xúc.


In [4]:
# --- CELL 3: MODEL CONFIG ---
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, 
    num_labels=NUM_LABELS, # Bây giờ là 7
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

# Config LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model = model.to(device)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,629,639 || all params: 357,996,558 || trainable%: 0.7345


In [7]:
# --- CELL 4: TRAINING (WITH EARLY STOPPING) ---
from transformers import EarlyStoppingCallback # <--- Import Callback

# 1. Data Collator (Giữ nguyên)
def data_collator(features):
    batch = tokenizer.pad(features, padding=True, return_tensors="pt")
    batch["labels"] = batch["labels"].float() 
    return batch

# 2. Metrics (Giữ nguyên)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    predictions = (probs > 0.5).astype(int)
    
    f1_micro = f1_score(y_true=labels, y_pred=predictions, average='micro')
    f1_macro = f1_score(y_true=labels, y_pred=predictions, average='macro')
    return {"f1_micro": f1_micro, "f1_macro": f1_macro}

# 3. Training Arguments (Có thay đổi)
training_args = TrainingArguments(
    output_dir="./results_ekman",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    
    # --- THAY ĐỔI QUAN TRỌNG CHO EARLY STOPPING ---
    num_train_epochs=15,             # Đặt số lớn (10-20) để model thoải mái học
    load_best_model_at_end=True,     # Bắt buộc True để quay về model tốt nhất sau khi dừng
    metric_for_best_model="f1_macro",# Dựa vào chỉ số này để quyết định dừng
    greater_is_better=True,          # F1 càng cao càng tốt
    evaluation_strategy="epoch",     # Kiểm tra sau mỗi epoch
    save_strategy="epoch",           # Lưu sau mỗi epoch (phải khớp với eval)
    save_total_limit=2,              # Chỉ giữ lại 2 checkpoint gần nhất cho nhẹ máy
    # -----------------------------------------------
    
    weight_decay=0.01,
    fp16=True,
    report_to="none"
)

# 4. Trainer (Thêm callback)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    
    # --- KÍCH HOẠT EARLY STOPPING ---
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)] 
    # patience=3: Nếu sau 3 epoch mà F1-Macro không tăng, thì dừng ngay lập tức.
)

print("🚀 Bắt đầu train Ekman Emotion Model (kèm Early Stopping)...")
trainer.train()

# 5. Lưu model
save_path = "best_ekman_model"
trainer.save_model(save_path)
print(f"🎉 Đã lưu model tốt nhất tại: {save_path}")

🚀 Bắt đầu train Ekman Emotion Model (kèm Early Stopping)...


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,0.213000,0.213894,0.682054,0.594695
2,0.208700,0.214126,0.688050,0.602502
3,0.205800,0.210345,0.684225,0.609854
4,0.206600,0.207212,0.690663,0.605274
5,0.203700,0.211266,0.689053,0.611453
6,0.197300,0.206496,0.698120,0.602944
7,0.197800,0.208414,0.695261,0.608243
8,0.194400,0.210517,0.688700,0.595247


🎉 Đã lưu model tốt nhất tại: best_ekman_model


In [8]:
# --- CELL 5: DOWNLOAD (KAGGLE) ---
import shutil
import os
from IPython.display import FileLink

# 1. Nén folder
print("📦 Đang nén folder model...")
shutil.make_archive("ekman_model_final", 'zip', "best_ekman_model")

# 2. Tạo link tải
print("✅ Đã nén xong! Hãy click vào link bên dưới để tải về:")
FileLink(r'ekman_model_final.zip')

📦 Đang nén folder model...
✅ Đã nén xong! Hãy click vào link bên dưới để tải về:


/kaggle/working/ekman_model_final.zip

In [1]:
# --- CELL 1: SETUP (FIXED VERSION) ---
# Gỡ bản lỗi và cài bản ổn định
!pip uninstall -y datasets transformers
!pip install -q datasets==2.19.0 soundfile librosa transformers==4.44.2 peft==0.12.0 accelerate scikit-learn torch torchaudio

import torch
import numpy as np
import warnings
# Import thư viện
import datasets
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🟢 Device: {device}")
print(f"✅ Datasets version: {datasets.__version__} (Bản này dùng SoundFile ổn định)")

Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1
Found existing installation: transformers 4.53.3
Uninstalling transformers-4.53.3:
  Successfully uninstalled transformers-4.53.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 11.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 96.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 67.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s e

2025-12-06 15:20:34.950458: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765034435.167030      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765034435.230635      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

🟢 Device: cuda
✅ Datasets version: 2.19.0 (Bản này dùng SoundFile ổn định)


In [3]:
# --- CELL 2: AUDIO PREPARATION (FINAL CHECK) ---
import datasets
import soundfile as sf
import warnings

# 1. KIỂM TRA PHIÊN BẢN (Chặn lỗi ngay từ đầu)
print(f"🔹 Datasets Version: {datasets.__version__}")
print(f"🔹 SoundFile Version: {sf.__version__}")

if datasets.__version__ != "2.19.0":
    raise RuntimeError("⚠️ LỖI: Bạn chưa Restart Session! Hãy vào Menu Runtime -> Restart Session rồi chạy lại Cell này.")

# 2. Config & Import
from datasets import load_dataset, Audio
from transformers import AutoFeatureExtractor
import numpy as np
warnings.filterwarnings("ignore")

# Cấu hình backend âm thanh
datasets.config.AUDIO_ENGINE = "soundfile"

# Mapping
IEMOCAP_STRING_MAPPING = {
    "ang": "anger", "fru": "anger", 
    "hap": "joy", "exc": "joy",    
    "sad": "sadness", "fea": "fear",
    "sur": "surprise", "dis": "disgust",
    "neu": "neutral", "oth": "neutral", "xxx": "neutral"
}
TARGET_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
label2id = {l: i for i, l in enumerate(TARGET_LABELS)}

# 3. Load Data & Process
print("⏳ Đang tải dataset IEMOCAP...")
dataset = load_dataset("AbstractTTS/IEMOCAP") 

print("⏳ Đang tải Feature Extractor...")
audio_checkpoint = "facebook/wav2vec2-base" 
feature_extractor = AutoFeatureExtractor.from_pretrained(audio_checkpoint)

# Cast column audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

def preprocess_audio(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays, 
        sampling_rate=16000, 
        max_length=16000 * 6, 
        truncation=True, 
        padding=True
    )
    new_labels = [label2id[IEMOCAP_STRING_MAPPING.get(l.strip(), "neutral")] for l in examples["major_emotion"]]
    inputs["labels"] = new_labels
    return inputs

print("⚙️ Đang xử lý Audio...")
# Batch size = 16 an toàn nhất
encoded_dataset = dataset.map(preprocess_audio, batched=True, batch_size=16, remove_columns=dataset['train'].column_names)

split_dataset = encoded_dataset["train"].train_test_split(test_size=0.2, seed=42)
print(f"✅ XONG! Train: {len(split_dataset['train'])} | Test: {len(split_dataset['test'])}")

🔹 Datasets Version: 2.19.0
🔹 SoundFile Version: 0.13.1
⏳ Đang tải dataset IEMOCAP...


Generating train split:   0%|          | 0/10039 [00:00<?, ? examples/s]

⏳ Đang tải Feature Extractor...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

⚙️ Đang xử lý Audio...


Map:   0%|          | 0/10039 [00:00<?, ? examples/s]

✅ XONG! Train: 8031 | Test: 2008


In [4]:
# --- CELL 3: AUDIO MODEL CONFIG (FIXED) ---
from transformers import AutoModelForAudioClassification
import torch

# 1. Khai báo lại tên model cho chắc chắn
audio_checkpoint = "facebook/wav2vec2-base" 
NUM_LABELS = 7 # Ekman có 7 cảm xúc

print(f"⏳ Đang tải model Audio: {audio_checkpoint}...")
model = AutoModelForAudioClassification.from_pretrained(
    audio_checkpoint,
    num_labels=NUM_LABELS,
    label2id=label2id, # Biến này đã có từ Cell 2
    id2label={v: k for k, v in label2id.items()} # Tạo ngược lại id2label
)

# 2. Đóng băng (Freeze) Feature Encoder
# Wav2Vec2 rất nặng, nếu train hết sẽ tràn VRAM ngay lập tức.
# Chúng ta chỉ train phần "Classification Head" ở cuối thôi.
model.freeze_feature_encoder()

# 3. Đẩy sang GPU
model.to(device)
print("✅ Audio Model đã sẵn sàng trên GPU!")

⏳ Đang tải model Audio: facebook/wav2vec2-base...


pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Audio Model đã sẵn sàng trên GPU!


In [5]:
# --- CELL 4: AUDIO TRAINING ---
from transformers import TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

# 1. Metric tính điểm (Accuracy + F1)
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

# 2. Cấu hình Training
training_args = TrainingArguments(
    output_dir="./results_iemocap_audio",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,          # Audio cần learning rate nhỏ hơn Text chút
    per_device_train_batch_size=16, # Nếu GPU yếu thì giảm xuống 8
    per_device_eval_batch_size=16,
    num_train_epochs=10,         # Audio cần nhiều epoch hơn Text (khoảng 10-15)
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,          # Chỉ giữ 2 model tốt nhất
    report_to="none",
    fp16=True                    # Tăng tốc GPU
)

# 3. Khởi tạo Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    tokenizer=feature_extractor, # Audio dùng feature_extractor thay cho tokenizer
    compute_metrics=compute_metrics,
)

print("🚀 Bắt đầu train Audio Model...")
trainer.train()

# 4. Lưu model
trainer.save_model("best_audio_model_iemocap")
print("🎉 Đã lưu Audio Model thành công!")

🚀 Bắt đầu train Audio Model...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.275527,0.888446,0.890327
2,0.403400,0.269193,0.887450,0.892151
3,0.403400,0.281320,0.900896,0.896199
4,0.307500,0.372193,0.873008,0.879544
5,0.307500,0.256726,0.875000,0.883480
6,0.279200,0.251283,0.895916,0.897390
7,0.279200,0.245150,0.903386,0.901597
8,0.258300,0.235178,0.903884,0.900259
9,0.258300,0.231547,0.909363,0.907088
10,0.229800,0.236704,0.895418,0.898094


🎉 Đã lưu Audio Model thành công!


In [6]:
# --- CELL 5: DOWNLOAD (KAGGLE) ---
import shutil
import os
from IPython.display import FileLink

# 1. Nén folder
print("📦 Đang nén folder model...")
shutil.make_archive("final_iemocap_audio", 'zip', "best_audio_model_iemocap")

# 2. Tạo link tải
print("✅ Đã nén xong! Hãy click vào link bên dưới để tải về:")
FileLink(r'final_iemocap_audio.zip')

📦 Đang nén folder model...
✅ Đã nén xong! Hãy click vào link bên dưới để tải về:


/kaggle/working/final_iemocap_audio.zip